# PolyRoute — Kaggle Execution Notebook

**Multilingual Speech Routing via Dual-Signal Language ID**  
GPU: T4 x2  |  Run cells top-to-bottom in order.

| Step | What it does | Est. time |
|---|---|---|
| 0 | Env setup, datasets version fix | <1 min |
| 1 | Download & cache all models (~10 GB) | ~15 min |
| 2 | Pipeline smoke test (silence → chain) | ~3 min |
| 3 | LID evaluation (30 langs × 50 samples) | ~20 min |
| 4 | Full ASR evaluation (30 langs × 30 samples) | ~40 min |
| 5 | Baselines B1-B4 | ~60 min |
| 6 | Learned routing policy | ~30 min |
| 6b | CER-oracle routing policy (~5 min, reuses Step 4) | ~5 min |
| 7 | Ablations A1-A9 | ~135 min |


## Step 0 — Environment Setup

Run this cell first. If it prints **'Restart required'**, go to **Kernel → Restart & Run All**.

In [ ]:
# ── Step 0: Environment Setup ────────────────────────────────────────────
import sys, os, pathlib, subprocess, importlib.metadata

# Find the repo root (directory that contains src/)
def _find_repo():
    # Check explicit known paths first
    candidates = [
        '/kaggle/working/LID-Router',
        '/kaggle/input/polyroute',
        '/kaggle/input/end-sem-project',
    ]
    for d in candidates:
        if os.path.isdir(os.path.join(d, 'src')):
            return d
    # Walk both /kaggle/working and /kaggle/input subdirectories
    for base in ['/kaggle/working', '/kaggle/input']:
        if os.path.isdir(base):
            for name in sorted(os.listdir(base)):
                p = os.path.join(base, name)
                if os.path.isdir(os.path.join(p, 'src')):
                    return p
    raise RuntimeError('Cannot find repo root. Make sure your dataset is attached and contains src/.')

REPO = _find_repo()
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)
print('Repo root:', REPO)
print('Working dir:', os.getcwd())

# datasets version check (google/fleurs needs < 4.0)
ver = importlib.metadata.version('datasets')
major = int(ver.split('.')[0])
print('datasets version:', ver)
if major >= 4:
    print('Downgrading datasets to 2.20.0 (required for google/fleurs)...')
    subprocess.run(['pip', 'install', 'datasets==2.20.0',
                    '--quiet', '--force-reinstall'], check=True)
    print('**** Restart required. Go to Kernel -> Restart & Run All ****')
else:
    print('OK: datasets version compatible')

# Install any missing packages
subprocess.run(['pip', 'install', 'openai-whisper', 'speechbrain',
                'jiwer', '--quiet'], check=True)
print('OK: extra packages installed')


## Step 1 — Download & Cache Models (~15 min, ~10 GB)

Run once per Kaggle session. Models are cached in `/root/.cache/huggingface/`.

In [ ]:
# ── Step 1: Download & cache all models ──────────────────────────────────
from transformers import Wav2Vec2ForSequenceClassification, AutoFeatureExtractor
from transformers import Wav2Vec2ForCTC, AutoProcessor
import whisper

# MMS-LID-4017  (~4 GB)
print('Downloading MMS-LID-4017...')
AutoFeatureExtractor.from_pretrained('facebook/mms-lid-4017')
Wav2Vec2ForSequenceClassification.from_pretrained('facebook/mms-lid-4017')
print('OK: MMS-LID-4017')

# MMS-1b-all ASR  (~4 GB)
print('Downloading MMS-1b-all...')
AutoProcessor.from_pretrained('facebook/mms-1b-all')
Wav2Vec2ForCTC.from_pretrained('facebook/mms-1b-all')
print('OK: MMS-1b-all')

# Whisper large-v3  (~3 GB)
print('Downloading Whisper large-v3...')
whisper.load_model('large-v3')
print('OK: Whisper large-v3')

print('All models cached.')


## Step 2 — Pipeline Smoke Test

Loads all models and runs a silent audio clip through the full chain. Should complete in ~3 min.

In [ ]:
# ── Step 2: Pipeline smoke test ──────────────────────────────────────────
import numpy as np
from src.pipeline import Pipeline

pipe = Pipeline()
pipe.load_models()  # ~2-3 min on T4

# Silence -> just checks the chain doesn't crash
dummy = np.zeros(16000, dtype=np.float32)
out = pipe.run(dummy)
print('language=%s, mode=%s, backend=%s' % (out.detected_language, out.routing_mode, out.backend_used))

pipe.unload_models()
print('OK: smoke test passed')


## Step 3 — LID Evaluation (30 langs × 50 samples ≈ 1500 utterances, ~20 min)

In [ ]:
# ── Step 3: LID-only evaluation ──────────────────────────────────────────
from src.pipeline import Pipeline
from evaluation.evaluate import evaluate_lid_only
from evaluation.data_loader import SUBSET_30_FLEURS
import json, pathlib

pipe = Pipeline()
pipe.load_models()

results = evaluate_lid_only(
    pipeline=pipe,
    lang_codes=SUBSET_30_FLEURS,
    split='validation',
    max_per_lang=50,
)
print('LID Accuracy: %.3f' % results.lid_accuracy())
print('Top-3 Accuracy: %.3f' % results.lid_top3_accuracy())

pathlib.Path('results').mkdir(exist_ok=True)
with open('results/lid_only_results.json', 'w') as f:
    json.dump(results.to_dict(), f, indent=2)
print('Saved results/lid_only_results.json')

pipe.unload_models()


## Step 4 — Full ASR Evaluation (30 langs × 30 samples, ~40 min)

In [ ]:
# ── Step 4: Full pipeline evaluation ─────────────────────────────────────
from evaluation.evaluate import evaluate_full
from evaluation.data_loader import SUBSET_30_FLEURS

pipe = Pipeline()
pipe.load_models()

results = evaluate_full(
    pipeline=pipe,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/eval_results.json',
)
print('CER: %.3f' % results.mean_cer())
print('WER: %.3f' % results.mean_wer())
print('Routing distribution:', results.routing_distribution())

pipe.unload_models()


## Step 5 — Baselines B1-B4 (~60 min total)

In [ ]:
# ── Step 5: Baselines ────────────────────────────────────────────────────
from evaluation.evaluate import (
    evaluate_baseline_oracle,
    evaluate_baseline_whisper_auto,
    evaluate_baseline_static_mms,
    evaluate_baseline_static_sb_whisper,
)
from evaluation.data_loader import SUBSET_30_FLEURS
import json

# B1: Oracle (upper bound)
r_b1 = evaluate_baseline_oracle(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)
# B2: Whisper auto-detect (no LID)
r_b2 = evaluate_baseline_whisper_auto(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)
# B3: Always MMS-1b-all
r_b3 = evaluate_baseline_static_mms(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)
# B4: SpeechBrain LID + Whisper
r_b4 = evaluate_baseline_static_sb_whisper(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)

# ── Step 5 FIX: Save baseline results (already computed above) ──

for name, r in [('B1_oracle', r_b1), ('B2_whisper_auto', r_b2),
                ('B3_static_mms', r_b3), ('B4_sb_whisper', r_b4)]:
    with open('results/%s.json' % name, 'w') as f:
        json.dump(r, f, indent=2)                    # r is already a dict
    overall_cer = r.get('overall', float('nan'))
    print('%s: CER=%.3f' % (name, overall_cer))


## Step 6 — Learned Routing Policy (~30 min)

In [ ]:
# ── Step 6: Learned routing policy ───────────────────────────────────────
from src.routing.policy_learned import generate_oracle_labels, LearnedRoutingPolicy
from evaluation.data_loader import load_fleurs, iterate_fleurs, SUBSET_30_FLEURS
from src.pipeline import Pipeline
import numpy as np, pathlib

pipe = Pipeline()
pipe.load_models()

fused_probs_list, uncertainty_list, true_langs = [], [], []
datasets = load_fleurs(SUBSET_30_FLEURS, split='validation', streaming=True)
for audio, sr, fleurs_code, true_lang, ref_text in iterate_fleurs(datasets, max_per_lang=50):
    lid_out = pipe.run_lid_only(audio, sr)
    fused_probs_list.append(lid_out['fused_probs'])
    uncertainty_list.append(lid_out['uncertainty'])
    true_langs.append(true_lang)

pipe.unload_models()

X, y = generate_oracle_labels(fused_probs_list, uncertainty_list, true_langs)
print('Training data: %s, label distribution: %s' % (X.shape, str(np.bincount(y))))

policy = LearnedRoutingPolicy()
# Improved architecture: input_dim auto-inferred from X.shape (11), BatchNorm + cosine LR, 100 epochs
history = policy.train_policy(X, y, epochs=100, lr=0.001)
print('Final val_acc: %.3f' % history['val_acc'][-1])

pathlib.Path('models').mkdir(exist_ok=True)
policy.save('models/routing_policy.pt')
print('Saved models/routing_policy.pt')

## Step 6b — CER-Oracle Routing Policy (~5 min)

Trains a second policy where routing labels come from **actual CER** on the test set
rather than LID accuracy. Reuses Step 4 output — no re-run needed.

Labels: CER < 0.15 → Mode A, CER < 0.35 → Mode B, CER ≥ 0.35 → Mode C

In [ ]:
# ── Step 6b: CER-oracle routing policy (reuses Step 4 results) ──────────
import json, pathlib
import numpy as np
from src.routing.policy_learned import LearnedRoutingPolicy

with open('results/eval_results.json') as f:
    data = json.load(f)

records = data.get('records', [])
if not records:
    raise RuntimeError('No per-utterance records in eval_results.json — '
                       'make sure Step 4 ran with the updated code.')

X_cer, y_cer = [], []
for rec in records:
    uvec = rec.get('uncertainty_vec')
    if uvec is None:
        continue
    cer = rec['cer']
    # CER-oracle labels: low CER → Mode A, mid → Mode B, high → Mode C
    label = 0 if cer < 0.15 else (1 if cer < 0.35 else 2)
    X_cer.append(uvec)
    y_cer.append(label)

X_cer = np.array(X_cer)
y_cer = np.array(y_cer)
print('CER-oracle training data: %s, label distribution: %s' % (X_cer.shape, str(np.bincount(y_cer))))

policy_cer = LearnedRoutingPolicy()
history_cer = policy_cer.train_policy(X_cer, y_cer, epochs=50, lr=0.001)
print('Final val_acc: %.3f' % history_cer['val_acc'][-1])

pathlib.Path('models').mkdir(exist_ok=True)
policy_cer.save('models/routing_policy_cer.pt')
print('Saved models/routing_policy_cer.pt')

## Step 7 — Ablations A1-A9 (~135 min)

A1–A8: component ablations. A9: CER-oracle policy comparison (requires Step 6b to complete first).
A6 rule-based half reuses Step 4 output (~40 min saved).

**Resume safe:** re-running this cell skips any ablations whose output JSON already exists in `results/ablations/`.

In [ ]:
# ── Step 7: Ablation study A1-A9 (with resume support) ──────────────────
import pathlib, json
from evaluation.ablation import run_all_ablations

SAVE_DIR = 'results/ablations'

# Check which ablations are already done
_output_files = {
    'A1': 'a1_mms_lid_only.json',
    'A2': 'a2_whisper_lid_only.json',
    'A3': 'a3_ecapa_lid.json',
    'A4': 'a4_no_routing.json',
    'A5': 'a5_no_confusion.json',
    'A6': 'a6_learned_policy.json',
    'A7': 'a7_mms_only_asr.json',
    'A8': 'a8_whisper_only_asr.json',
    'A9': 'a9_cer_oracle_policy.json',
}

done = [k for k, v in _output_files.items()
        if (pathlib.Path(SAVE_DIR) / v).exists()]
remaining = [k for k in _output_files if k not in done]

print('Already completed:', done if done else 'None')
print('Will run:', remaining if remaining else 'None — all done!')

if remaining:
    # run_all_ablations auto-skips completed runs via file checks
    run_all_ablations(max_per_lang=30, save_dir=SAVE_DIR)
    print('Ablation study complete. Results in', SAVE_DIR)
else:
    print('All ablations already complete. Results in', SAVE_DIR)

## Step 8 — Track 3: Confidence-Weighted MMS Fallback (~80 min)

**What changed (zero-compute logic fix):**
1. `backend_selector.py` now uses a per-language quality table (`config/backend_quality.yaml`) instead of always picking Whisper for Mode A.
2. Only 6/28 languages genuinely prefer Whisper (arb, cmn, hin, jpn, srp, urd). The other 22 use MMS.
3. When LID confidence < 0.5, backend defaults to MMS (degrades more gracefully on wrong-language input).
4. Mode B multi-hypothesis now tries **both** backends for the top-1 candidate and keeps the better one.

**Expected impact:** CER should drop significantly (target: beat B3's 0.132).

In [ ]:
# ── Step 8 Setup: Reload Track 3 modules ────────────────────────────────
# Run this cell once before running Step 8a or Step 8b (or both).

import json, importlib
from evaluation.evaluate import evaluate_full
from evaluation.data_loader import SUBSET_30_FLEURS

import src.asr.backend_selector as _bs
import src.decoding.single_decode as _sd
import src.decoding.multi_hypothesis as _mh
importlib.reload(_bs)
importlib.reload(_sd)
importlib.reload(_mh)
_bs._backend_prefs = None   # reset singleton so YAML is re-read

print('Track 3 modules reloaded.')


## Step 8a — Rules Policy + Track 3 (~40 min)

Re-evaluates the **rules-based routing policy** with the Track 3 confidence-weighted backend selector.  
Saves to `results/step8_track3_rules.json`.

In [ ]:
# ── Step 8a: Rules-based policy with Track 3 backend selector ──────────
from src.pipeline import Pipeline

pipe_rules = Pipeline(routing_policy='rules')
pipe_rules.load_models()

results_rules = evaluate_full(
    pipeline=pipe_rules,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/step8_track3_rules.json',
)
print('\n=== Step 8a: Track 3 + Rules Policy ===')
print('CER: %.3f' % results_rules.mean_cer())
print('WER: %.3f' % results_rules.mean_wer())
print('Routing:', results_rules.routing_distribution())

pipe_rules.unload_models()


## Step 8b — Learned Policy + Track 3 (~40 min)

Re-evaluates the **learned MLP routing policy** (from Step 6) with the Track 3 confidence-weighted backend selector.  
> **Requires:** `models/routing_policy.pt` (trained in Step 6).  

Saves to `results/step8_track3_learned.json`.

In [ ]:
# ── Step 8b: Learned MLP policy with Track 3 backend selector ──────────
from src.pipeline import Pipeline

pipe_learned = Pipeline(routing_policy='learned')
pipe_learned.routing_agent.load_learned_policy('models/routing_policy.pt')
pipe_learned.load_models()

results_learned = evaluate_full(
    pipeline=pipe_learned,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/step8_track3_learned.json',
)
print('\n=== Step 8b: Track 3 + Learned Policy ===')
print('CER: %.3f' % results_learned.mean_cer())
print('WER: %.3f' % results_learned.mean_wer())
print('Routing:', results_learned.routing_distribution())

pipe_learned.unload_models()


## Step 8 — Comparison Table

Run after both 8a and 8b have completed (reads from saved JSON files).

In [ ]:
# ── Step 8 Comparison: Track 3 vs previous baselines ────────────────────
import json

def _load_result(path):
    try:
        with open(path) as f: d = json.load(f)
        return d.get('overall_mean_cer', d.get('overall', float('nan'))), \
               d.get('overall_mean_wer', float('nan'))
    except Exception:
        return float('nan'), float('nan')

print('=' * 65)
print('COMPARISON: Track 3 vs Previous Results')
print('=' * 65)
print(f'{"System":<35} {"CER":>8} {"WER":>8}')
print('-' * 65)
for name, path in [
    ('B3 Static MMS',       'results/B3_static_mms.json'),
    ('A6 Learned (old)',    'results/ablations/a6_learned_policy.json'),
    ('Step4 Rules (old)',   'results/eval_results.json'),
]:
    cer, wer = _load_result(path)
    print(f'{name:<35} {cer:>8.3f} {wer:>8.3f}')
print('-' * 65)
cer_r, wer_r = _load_result('results/step8_track3_rules.json')
cer_l, wer_l = _load_result('results/step8_track3_learned.json')
print(f'{"Track 3 + Rules (NEW)":<35} {cer_r:>8.3f} {wer_r:>8.3f}')
print(f'{"Track 3 + Learned (NEW)":<35} {cer_l:>8.3f} {wer_l:>8.3f}')
print('=' * 65)
try:
    with open('results/step8_track3_learned.json') as f: d = json.load(f)
    per_lang = d.get('per_language', {})
    if per_lang:
        print('\nPer-language CER (Track 3 + Learned):')
        for lang in sorted(per_lang):
            print(f'  {lang}: CER={per_lang[lang]["mean_cer"]:.3f}')
except Exception as e:
    print(f'(Per-language breakdown unavailable: {e})')
